**Hybrid RAG implementation**

In [1]:
#installing libraries

!pip install langchain-google-genai
!pip install sentence-transformers langchain-huggingface
!pip install langchain-community
!pip install langchain-Chroma
!pip install pypdf
!pip install ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 32.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-co

In [3]:
!pip install langchain-pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [3]:
!pip install langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 11.0 MB/s eta 0:00:00


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_chroma import Chroma
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.retrievers import BM25Retriever
from pinecone import Pinecone
from pydantic import BaseModel, Field
from sentence_transformers import CrossEncoder
import bs4

from dotenv import load_dotenv
load_dotenv()
from google.colab import userdata
import os

In [9]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 56.4 MB/s eta 0:00:00


In [10]:
# PDF Loader
docs = PyMuPDFLoader("/content/ARBL Annual-Report-2013-14.pdf").load()

#Semantic chunking
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
chunks = SemanticChunker(embeddings).split_documents(docs)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [28]:
from pinecone import Pinecone, PodSpec
from google.colab import userdata
import os
from pinecone import Pinecone,ServerlessSpec

pc = Pinecone(api_key=userdata.get("PINE"))
index_name = "fin-rag"

# Define the correct dimension for the index based on the embedding model
expected_dimension = 768  # BAAI/bge-base-en-v1.5 has 768 dimensions

# Check if index exists, and create if it doesn't
if index_name not in pc.list_indexes():
    print(f"Creating new Pinecone index: {index_name} with dimension {expected_dimension}")
    # IMPORTANT: You need to replace 'your-pinecone-environment' with the exact environment name from your Pinecone dashboard.
    # For AWS free-tier, it's often 'us-east-1-aws'.
    pc.create_index(
        index_name,
        dimension=expected_dimension,
        metric='cosine',
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
else:
    print(f"Pinecone index '{index_name}' already exists.")

index = pc.Index(index_name)

# Store embeddings in Pinecone
for i, doc in enumerate(chunks):
    index.upsert(
        vectors=[{
            "id": str(i),
            "values": embeddings.embed_query(doc.page_content),
            "metadata": {
                "text": doc.page_content,
                "page": doc.metadata.get("page", 0)
            }
        }]
    )

Creating new Pinecone index: fin-rag with dimension 768


In [35]:
!pip install rank_bm25

In [67]:
#Hybrid search : vector + BM25

bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 5

def hybrid_search(query,k=5):
  vector = embeddings.embed_query(query)

  pinecone_result = index.query(
      vector = vector,
      top_k = k,
      include_metadata = True
  )

  bm25_result= bm25.invoke(query)

  return [
      r["metadata"]["text"]
      for r in pinecone_result["matches"]
  ] + [
      r.page_content
      for r in bm25_result
  ]

In [68]:
#Re-ranking

reranker = CrossEncoder("BAAI/bge-reranker-base")

def rerank(query,docs,k=5):
  scores = reranker.predict([(query,d) for d in docs])
  return [d for score, d in sorted(zip(scores,docs),reverse=True)[:k]]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [75]:
#Structured output

class Answer(BaseModel):
  answer:str
  financial_year:str
  source_pages:list[int]

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0,api_key=userdata.get("GEMINI")).with_structured_output(Answer)

In [76]:
#Refine chain

def refine_chain(query,docs):
  answer = llm.invoke(
      f""" answer using the provide financial context.
      "Question": {query}
      "context": {docs[0]}
      """
  )

  for doc in docs[1:]:
    answer = llm.invoke(f""" Refine the existing answer using the additional context)
    Question:{query}
    Existing answer: {answer}
    Additional context:{docs}""")
    return answer

In [77]:
# Complete RAG

def financial_rag(query):
  docs = hybrid_search(query)
  docs = rerank(query,docs)

  return refine_chain(query,docs)

#Example

result = financial_rag("What was the net profit FY2013-14?")
print(result)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


answer="For FY2013-14, the Net Profit (Profit After Tax / Profit for the year) was ₹3,674.36 million (reported as ₹3,674 million in the Directors' Report), representing a 28% growth compared to ₹2,867.05 million in FY2012-13. The Profit Before Tax (PBT) for FY2013-14 was ₹5,366.70 million (reported as ₹5,367 million)." financial_year='FY2013-14' source_pages=[]


In [81]:
print(result.answer)

For FY2013-14, the Net Profit (Profit After Tax / Profit for the year) was ₹3,674.36 million (reported as ₹3,674 million in the Directors' Report), representing a 28% growth compared to ₹2,867.05 million in FY2012-13. The Profit Before Tax (PBT) for FY2013-14 was ₹5,366.70 million (reported as ₹5,367 million).
